## Phase 2: Performance Metrics

This notebook continues the analysis completed in Phase 1.

The objective is to compare the performance of Vanguard’s redesigned interface with the traditional interface using descriptive KPIs and to explore where differences occur within the user journey.

The analysis covers:

- **KPI 1:** Completion rate
- **Additional KPI:** Step-level conversion and drop-off
- **KPI 2:** Time spent between process steps
- **KPI 3:** Backward-transition rate (error proxy)

The additional funnel analysis examines where clients progress or drop off within the process and provides more actionable insights for potential UX improvements and future A/B tests.

The notebook also includes **Hypothesis 3**, which statistically tests whether step-level conversion differs between the Test and Control groups.

The required overall completion-rate hypotheses are handled separately in the main experiment evaluation.


In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.stats.proportion import proportions_ztest

In [12]:
# Exported from Notebook 1 after Phase 1 cleaning.
experiment_final_df = pd.read_csv("experiment_final_df.csv")

print("Shape:", experiment_final_df.shape)
print("Unique clients:", experiment_final_df["client_id"].nunique())

experiment_final_df.head()

Shape: (50487, 12)
Unique clients: 50487


,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,tenure_check_correct,Variation,furthest_step_reached
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0,6.0,Test,confirm
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0,7.0,Control,confirm
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0,5.0,Test,step_3
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0,16.0,Test,start
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0,12.0,Control,start


### Validate available columns

Before calculating completion rate, we confirm the exact column names in the client-level dataframe.

This prevents errors caused by using a column name that does not exist or has different capitalization.

In [13]:
# Display all column names in the Phase 1 client-level dataframe
print(experiment_final_df.columns.tolist())

['client_id', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth', 'tenure_check_correct', 'Variation', 'furthest_step_reached']


## KPI 1 – Completion Rate

Completion rate is calculated at client level.

A client is classified as completed when `furthest_step_reached` equals `confirm`.

This ensures each client is counted once, regardless of how many visits or web events they generated.

In [14]:
# Create a client-level completion indicator
client_completion_df = experiment_final_df[
    ["client_id", "Variation", "furthest_step_reached"]
].copy()

client_completion_df["completed"] = (
    client_completion_df["furthest_step_reached"] == "confirm"
)

client_completion_df.head()

,client_id,Variation,furthest_step_reached,completed
0,836976,Test,confirm,True
1,2304905,Control,confirm,True
2,1439522,Test,step_3,False
3,1562045,Test,start,False
4,5126305,Control,start,False


### Completion indicator check

The `completed` column correctly identifies whether each client reached the `confirm` step.

Clients with `confirm` are marked `True`. Clients whose furthest step was `start`, `step_1`, `step_2` or `step_3` are marked `False`.

In [15]:
# Summarize completion by experiment group
completion_summary = (
    client_completion_df
    .groupby("Variation")
    .agg(
        total_clients=("client_id", "nunique"),
        completed_clients=("completed", "sum")
    )
)

completion_summary["completion_rate"] = (
    completion_summary["completed_clients"]
    / completion_summary["total_clients"]
)

completion_summary["completion_rate_percent"] = (
    completion_summary["completion_rate"] * 100
)

completion_summary

,total_clients,completed_clients,completion_rate,completion_rate_percent
Variation,,,,
Control,23526,15428,0.655785,65.578509
Test,26961,18682,0.692927,69.292682


In [16]:
# Calculate the practical improvement of Test over Control

test_completion_rate = completion_summary.loc["Test", "completion_rate"]
control_completion_rate = completion_summary.loc["Control", "completion_rate"]

absolute_difference = test_completion_rate - control_completion_rate
relative_improvement = absolute_difference / control_completion_rate

print(f"Control completion rate: {control_completion_rate:.2%}")
print(f"Test completion rate: {test_completion_rate:.2%}")
print(f"Absolute difference: {absolute_difference:.2%}")
print(f"Relative improvement: {relative_improvement:.2%}")

Control completion rate: 65.58%
Test completion rate: 69.29%
Absolute difference: 3.71%
Relative improvement: 5.66%


### Completion-rate interpretation

The Test group achieved a completion rate of 69.29%, compared with 65.58% for the Control group.

This is:

- an absolute improvement of 3.71 percentage points
- a relative improvement of 5.66%

The observed relative improvement exceeds Vanguard’s 5% cost-effectiveness threshold. Statistical testing is still required to determine whether this difference is unlikely to be due to chance.

## Additional KPI – Funnel Reach and Step-to-Step Drop-Off

Overall completion rate tells us whether clients reached the final `confirm` step, but it does not show where users stop progressing through the journey.

To make the analysis more actionable, we also calculate:

- how many clients reached each process step
- the conversion rate from one step to the next
- the drop-off rate between consecutive steps
- differences between the Test and Control groups

This helps identify which part of the journey contributes most to abandonment and which stage could be prioritized in a future A/B test or UX improvement.

In [17]:
# Define the expected order of the funnel

step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

funnel_client_df = experiment_final_df[
    ["client_id", "Variation", "furthest_step_reached"]
].copy()

funnel_client_df["furthest_step_number"] = (
    funnel_client_df["furthest_step_reached"].map(step_order)
)

funnel_client_df.head()

,client_id,Variation,furthest_step_reached,furthest_step_number
0,836976,Test,confirm,4
1,2304905,Control,confirm,4
2,1439522,Test,step_3,3
3,1562045,Test,start,0
4,5126305,Control,start,0


### Calculate funnel reach

A client is counted as having reached a step when their furthest recorded step is equal to or beyond that stage.

This creates a cumulative funnel for the Test and Control groups and shows how many clients remain at each stage of the process.

In [18]:
# Calculate the number and percentage of clients reaching each funnel stage

funnel_rows = []

for variation in ["Control", "Test"]:
    group = funnel_client_df[
        funnel_client_df["Variation"] == variation
    ]

    total_clients = group["client_id"].nunique()

    for step_name, step_number in step_order.items():
        clients_reached = (
            group["furthest_step_number"] >= step_number
        ).sum()

        funnel_rows.append({
            "Variation": variation,
            "process_step": step_name,
            "clients_reached": clients_reached,
            "reach_rate": clients_reached / total_clients
        })

funnel_summary = pd.DataFrame(funnel_rows)

funnel_summary

,Variation,process_step,clients_reached,reach_rate
0,Control,start,23526,1.000000
1,Control,step_1,20241,0.860367
2,Control,step_2,18786,0.798521
3,Control,step_3,17521,0.744750
4,Control,confirm,15428,0.655785
5,Test,start,26961,1.000000
6,Test,step_1,24507,0.908980
7,Test,step_2,22528,0.835577
8,Test,step_3,21118,0.783280
9,Test,confirm,18682,0.692927


### Calculate step-to-step conversion and drop-off

To understand where users discontinue the process, we calculate:

- the conversion rate between consecutive steps
- the corresponding drop-off rate

This provides more detailed insight than the overall completion rate and helps identify which stage could benefit most from future interface improvements.

In [19]:
# Calculate step-to-step conversion and drop-off

funnel_summary["previous_clients_reached"] = (
    funnel_summary
    .groupby("Variation")["clients_reached"]
    .shift(1)
)

funnel_summary["step_conversion_rate"] = (
    funnel_summary["clients_reached"]
    / funnel_summary["previous_clients_reached"]
)

funnel_summary["drop_off_rate"] = (
    1 - funnel_summary["step_conversion_rate"]
)

funnel_summary

,Variation,process_step,clients_reached,reach_rate,previous_clients_reached,step_conversion_rate,drop_off_rate
0,Control,start,23526,1.000000,NaN,NaN,NaN
1,Control,step_1,20241,0.860367,23526.0,0.860367,0.139633
2,Control,step_2,18786,0.798521,20241.0,0.928116,0.071884
3,Control,step_3,17521,0.744750,18786.0,0.932663,0.067337
4,Control,confirm,15428,0.655785,17521.0,0.880543,0.119457
5,Test,start,26961,1.000000,NaN,NaN,NaN
6,Test,step_1,24507,0.908980,26961.0,0.908980,0.091020
7,Test,step_2,22528,0.835577,24507.0,0.919248,0.080752
8,Test,step_3,21118,0.783280,22528.0,0.937411,0.062589
9,Test,confirm,18682,0.692927,21118.0,0.884648,0.115352


### Compare funnel performance between Test and Control

To make the funnel easier to interpret, we compare the step-level conversion and drop-off rates for both experiment groups side by side.

This highlights at which stage the redesigned interface performs better or worse than the original interface.

In [20]:
# Compare conversion and drop-off rates side by side

funnel_comparison = funnel_summary.pivot(
    index="process_step",
    columns="Variation",
    values=["step_conversion_rate", "drop_off_rate"]
)

funnel_comparison

step_conversion_rate           drop_off_rate          
Variation                 Control      Test       Control      Test
process_step                                                       
confirm                  0.880543  0.884648      0.119457  0.115352
start                         NaN       NaN           NaN       NaN
step_1                   0.860367  0.908980      0.139633  0.091020
step_2                   0.928116  0.919248      0.071884  0.080752
step_3                   0.932663  0.937411      0.067337  0.062589

### Compare differences between Test and Control

To identify where the redesign had the greatest impact, we calculate the difference in conversion and drop-off rates between the Test and Control groups for each funnel step.

In [21]:
# Create a clean comparison table

funnel_difference = pd.DataFrame({
    "Control Conversion": funnel_comparison["step_conversion_rate"]["Control"],
    "Test Conversion": funnel_comparison["step_conversion_rate"]["Test"],
    "Conversion Difference": (
        funnel_comparison["step_conversion_rate"]["Test"]
        - funnel_comparison["step_conversion_rate"]["Control"]
    ),
    "Control Drop-off": funnel_comparison["drop_off_rate"]["Control"],
    "Test Drop-off": funnel_comparison["drop_off_rate"]["Test"],
    "Drop-off Difference": (
        funnel_comparison["drop_off_rate"]["Test"]
        - funnel_comparison["drop_off_rate"]["Control"]
    )
})

funnel_difference

,Control Conversion,Test Conversion,Conversion Difference,Control Drop-off,Test Drop-off,Drop-off Difference
process_step,,,,,,
confirm,0.880543,0.884648,0.004105,0.119457,0.115352,-0.004105
start,NaN,NaN,NaN,NaN,NaN,NaN
step_1,0.860367,0.908980,0.048612,0.139633,0.091020,-0.048612
step_2,0.928116,0.919248,-0.008869,0.071884,0.080752,0.008869
step_3,0.932663,0.937411,0.004749,0.067337,0.062589,-0.004749


In [22]:
# Add clear transition labels for interpretation

transition_labels = {
    "step_1": "start → step_1",
    "step_2": "step_1 → step_2",
    "step_3": "step_2 → step_3",
    "confirm": "step_3 → confirm"
}

funnel_difference = funnel_difference.drop(index="start").copy()

funnel_difference["transition"] = (
    funnel_difference.index.map(transition_labels)
)

funnel_difference

,Control Conversion,Test Conversion,Conversion Difference,Control Drop-off,Test Drop-off,Drop-off Difference,transition
process_step,,,,,,,
confirm,0.880543,0.884648,0.004105,0.119457,0.115352,-0.004105,step_3 → confirm
step_1,0.860367,0.908980,0.048612,0.139633,0.091020,-0.048612,start → step_1
step_2,0.928116,0.919248,-0.008869,0.071884,0.080752,0.008869,step_1 → step_2
step_3,0.932663,0.937411,0.004749,0.067337,0.062589,-0.004749,step_2 → step_3


### Additional KPI Interpretation – Step-Level Conversion and Drop-Off

The funnel analysis shows that the Test group has a higher overall completion rate, but the redesign does not improve every transition equally.

The largest improvement occurs from `start → step_1`:

- Control conversion: 86.04%
- Test conversion: 90.90%
- Improvement: approximately 4.86 percentage points
- Test drop-off is approximately 4.86 percentage points lower

For `step_1 → step_2`, however, the Test group performs slightly worse:

- Control conversion: 92.81%
- Test conversion: 91.92%
- Test drop-off is approximately 0.89 percentage points higher

The remaining differences are smaller:

- `step_2 → step_3`: Test conversion is approximately 0.47 percentage points higher
- `step_3 → confirm`: Test conversion is approximately 0.41 percentage points higher

Descriptively, the largest contribution to the Test group's higher overall completion rate appears to occur at the beginning of the journey, while `step_1 → step_2` is the only transition where the redesigned experience shows higher drop-off.

These differences have not yet been statistically tested. The `step_1 → step_2` transition could therefore be investigated further as a potential focus for a follow-up A/B test or UX improvement.

## KPI 2 – Time Spent on Each Step

The client-level dataframe cannot be used for time analysis because it contains one row per client.

For this KPI, we use the original web activity data. Each row represents one event. Events are sorted chronologically within each `visit_id`.

Time spent on a step is estimated as the difference between the timestamp of the current event and the timestamp of the next event within the same visit.

In [23]:
# Load the two web activity files and the experiment assignment file

web_data_pt1_df = pd.read_csv(
    "data/df_final_web_data_pt_1.zip",
    compression="zip"
)

web_data_pt2_df = pd.read_csv(
    "data/df_final_web_data_pt_2.zip",
    compression="zip"
)

experiment_clients_df = pd.read_csv(
    "data/df_final_experiment_clients.txt"
)

print("Web part 1:", web_data_pt1_df.shape)
print("Web part 2:", web_data_pt2_df.shape)
print("Experiment clients:", experiment_clients_df.shape)

Web part 1: (343141, 5)
Web part 2: (412264, 5)
Experiment clients: (70609, 2)


### Create the event-level experiment dataset

The two web activity files are combined into a single event-level dataset.

This dataset is then merged with the experiment assignment file so that every event is labelled as either **Test** or **Control**.

Finally, the events are sorted chronologically within each visit, which is required to calculate the time between consecutive steps.

In [24]:
# Combine both web activity datasets
combined_web_data_df = pd.concat(
    [web_data_pt1_df, web_data_pt2_df],
    ignore_index=True
).drop_duplicates()

# Convert timestamps
combined_web_data_df["date_time"] = pd.to_datetime(
    combined_web_data_df["date_time"]
)

# Merge with experiment assignment
event_level_df = combined_web_data_df.merge(
    experiment_clients_df,
    on="client_id",
    how="inner"
)

# Keep only Test and Control
event_level_df = event_level_df[
    event_level_df["Variation"].isin(["Test", "Control"])
].copy()

# Sort events chronologically
event_level_df = event_level_df.sort_values(
    ["visit_id", "date_time"]
).reset_index(drop=True)

In [25]:
# Validate the event-level dataset

print("Rows:", len(event_level_df))
print("Unique clients:", event_level_df["client_id"].nunique())
print("Unique visits:", event_level_df["visit_id"].nunique())

print("\nVariation counts:")
print(event_level_df["Variation"].value_counts())

event_level_df.head()

Rows: 317235
Unique clients: 50500
Unique visits: 69205

Variation counts:
Variation
Test       176699
Control    140536
Name: count, dtype: int64


,client_id,visitor_id,visit_id,process_step,date_time,Variation
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,Test
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,Test
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,Test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,Test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,Test


### Calculate time between consecutive events

The time spent on a process step is estimated as the time difference between one event and the next event within the same visit.

The last event of a visit has no following event and is therefore excluded from the calculation.

In [26]:
# Calculate the next event within each visit

event_level_df["next_time"] = (
    event_level_df.groupby("visit_id")["date_time"].shift(-1)
)

event_level_df["next_step"] = (
    event_level_df.groupby("visit_id")["process_step"].shift(-1)
)

# Calculate elapsed time in seconds

event_level_df["time_seconds"] = (
    event_level_df["next_time"] - event_level_df["date_time"]
).dt.total_seconds()

# Remove the final event of each visit
time_df = event_level_df.dropna(subset=["time_seconds"]).copy()

# Remove negative durations if they exist
time_df = time_df[time_df["time_seconds"] >= 0]

time_df.head()

,client_id,visitor_id,visit_id,process_step,date_time,Variation,next_time,next_step,time_seconds
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,Test,2017-04-26 13:23:09,confirm,52.0
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,Test,2017-04-09 16:21:12,step_1,16.0
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,Test,2017-04-09 16:21:21,step_2,9.0
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,Test,2017-04-09 16:21:35,step_1,14.0
5,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:35,Test,2017-04-09 16:21:41,step_1,6.0


In [27]:
# Validate calculated durations

print("Transitions:", len(time_df))

print("\nSummary statistics:")
print(time_df["time_seconds"].describe())

Transitions: 248030

Summary statistics:
count    248030.000000
mean         84.207939
std         215.986776
min           0.000000
25%          13.000000
50%          36.000000
75%          83.000000
max       40235.000000
Name: time_seconds, dtype: float64


### Inspect extreme duration values before filtering

Before excluding any observations, we inspect unusually long gaps between consecutive events.

Durations above 30 minutes may represent inactivity or users leaving the process open rather than continuous interaction. The 30-minute threshold is an analytical assumption, so the extreme observations are reviewed before applying the filter.

In [28]:
# Inspect event gaps longer than 30 minutes

time_outliers_df = time_df[
    time_df["time_seconds"] > 1800
].copy()

print("Total valid transitions:", len(time_df))
print("Transitions > 30 minutes:", len(time_outliers_df))

print(
    f"Percentage > 30 minutes: "
    f"{len(time_outliers_df) / len(time_df) * 100:.2f}%"
)

time_outliers_df[
    [
        "client_id",
        "visit_id",
        "Variation",
        "process_step",
        "next_step",
        "time_seconds"
    ]
].sort_values(
    "time_seconds",
    ascending=False
).head(20)

Total valid transitions: 248030
Transitions > 30 minutes: 354
Percentage > 30 minutes: 0.14%


,client_id,visit_id,Variation,process_step,next_step,time_seconds
258400,2182225,831987489_84761163210_335684,Control,start,start,40235.0
132852,6392043,478417687_30092205462_548423,Test,start,start,24819.0
134725,9553121,483738263_4057680487_61869,Control,step_2,start,21763.0
101869,4647786,391696103_92230204739_479887,Test,confirm,confirm,14581.0
176525,8883167,602386199_13818409466_144534,Test,confirm,start,12980.0
185467,6420672,627596734_90499955288_891366,Control,start,start,11207.0
140846,9971962,501042989_50588313479_700675,Test,step_2,start,11204.0
171796,5282553,588904389_16165570985_286826,Control,step_1,start,10286.0
177527,2882702,605153028_51517094625_529507,Control,confirm,confirm,9396.0
110676,4803107,416635057_22840074620_580752,Test,step_1,start,8797.0


### Handling extreme time values

The duration inspection identified 354 transitions above 30 minutes, representing only 0.14% of all valid transitions.

Although these observations are rare, some gaps last several hours, with a maximum above 11 hours. These values are unlikely to represent continuous active interaction and may disproportionately affect the average.

For the primary time-spent analysis, durations above 30 minutes are excluded. The unfiltered data is retained separately, and median duration is also reported because it is less sensitive to extreme values.

In [29]:
# Remove unrealistic inactive sessions (>30 minutes)

time_filtered_df = time_df[
    time_df["time_seconds"] <= 1800
].copy()

print("Transitions before filtering:", len(time_df))
print("Transitions after filtering:", len(time_filtered_df))

time_filtered_df["time_seconds"].describe()

Transitions before filtering: 248030
Transitions after filtering: 247676


count    247676.000000
mean         79.987912
std         147.247367
min           0.000000
25%          13.000000
50%          36.000000
75%          82.000000
max        1800.000000
Name: time_seconds, dtype: float64

### Average and median time by process step

After excluding event gaps above 30 minutes, the average and median duration are calculated for each process step and experiment group.

The median is particularly important because time data remains right-skewed even after filtering.

In [31]:
# Summarize time spent by experiment group and process step

time_summary = (
    time_filtered_df
    .groupby(["Variation", "process_step"])
    .agg(
        transition_count=("time_seconds", "count"),
        average_seconds=("time_seconds", "mean"),
        median_seconds=("time_seconds", "median")
    )
    .reset_index()
)

time_summary["average_minutes"] = (
    time_summary["average_seconds"] / 60
)

time_summary["median_minutes"] = (
    time_summary["median_seconds"] / 60
)

time_summary

,Variation,process_step,transition_count,average_seconds,median_seconds,average_minutes,median_minutes
0,Control,confirm,1996,156.160321,49.0,2.602672,0.816667
1,Control,start,35676,60.911733,20.0,1.015196,0.333333
2,Control,step_1,26026,47.528433,20.0,0.792141,0.333333
3,Control,step_2,24305,90.214112,64.0,1.503569,1.066667
4,Control,step_3,20251,133.298306,72.0,2.221638,1.200000
5,Test,confirm,4172,221.972915,94.0,3.699549,1.566667
6,Test,start,46241,56.240912,14.0,0.937349,0.233333
7,Test,step_1,35500,58.481380,27.0,0.974690,0.450000
8,Test,step_2,29571,88.133949,61.0,1.468899,1.016667
9,Test,step_3,23938,124.832693,57.0,2.080545,0.950000



### Review the impact of the `confirm` step

The `confirm` step is the final stage of the process and has no required subsequent step.

Before deciding whether it should be included in the primary time-spent KPI, we compare the results with and without `confirm`.

This sensitivity check shows whether post-confirmation activity materially affects the overall time comparison between the Test and Control groups.

In [43]:
# Compare overall time results with and without the confirm step

with_confirm = (
    time_filtered_df
    .groupby("Variation")["time_seconds"]
    .agg(
        average_with_confirm="mean",
        median_with_confirm="median"
    )
)

without_confirm = (
    time_filtered_df[
        time_filtered_df["process_step"] != "confirm"
    ]
    .groupby("Variation")["time_seconds"]
    .agg(
        average_without_confirm="mean",
        median_without_confirm="median"
    )
)

confirm_impact = with_confirm.join(without_confirm)

confirm_impact["average_difference"] = (
    confirm_impact["average_without_confirm"]
    - confirm_impact["average_with_confirm"]
)

confirm_impact["median_difference"] = (
    confirm_impact["median_without_confirm"]
    - confirm_impact["median_with_confirm"]
)

confirm_impact.round(2)

,average_with_confirm,median_with_confirm,average_without_confirm,median_without_confirm,average_difference,median_difference
Variation,,,,,,
Control,79.57,37.0,78.13,37.0,-1.44,0.0
Test,80.31,35.0,75.94,34.0,-4.37,-1.0


### Sensitivity Check Interpretation

Including `confirm` affects the Test group more strongly than the Control group.

- Control mean: 79.57 seconds with `confirm` vs 78.13 seconds without it.
- Test mean: 80.31 seconds with `confirm` vs 75.94 seconds without it.
- Median times change very little.

With `confirm` included, the Test group appears slightly slower overall. After excluding `confirm`, the Test group has a lower overall mean time.

Because `confirm` is the final stage and has no required next step, these post-confirmation durations do not represent progression through the funnel.

### KPI 2 Data-Quality Adjustment

Based on the sensitivity check, `confirm` is excluded from the primary step-time KPI.

Durations starting from `confirm` represent activity after the user has already reached the final process stage, such as repeated confirmation events or later navigation.

The primary KPI therefore focuses on:

`start`, `step_1`, `step_2` and `step_3`

The `confirm` results are retained separately as a sensitivity check rather than discarded from the analysis.

In [44]:
# Create the final KPI 2 summary excluding confirm

time_kpi_df = time_summary[
    time_summary["process_step"] != "confirm"
].copy()

time_kpi_df

,Variation,process_step,transition_count,average_seconds,median_seconds,average_minutes,median_minutes
1,Control,start,35676,60.911733,20.0,1.015196,0.333333
2,Control,step_1,26026,47.528433,20.0,0.792141,0.333333
3,Control,step_2,24305,90.214112,64.0,1.503569,1.066667
4,Control,step_3,20251,133.298306,72.0,2.221638,1.200000
6,Test,start,46241,56.240912,14.0,0.937349,0.233333
7,Test,step_1,35500,58.481380,27.0,0.974690,0.450000
8,Test,step_2,29571,88.133949,61.0,1.468899,1.016667
9,Test,step_3,23938,124.832693,57.0,2.080545,0.950000


In [45]:
# Compare average and median time by process step

time_comparison = time_kpi_df.pivot(
    index="process_step",
    columns="Variation",
    values=["average_seconds", "median_seconds"]
)

time_comparison

average_seconds             median_seconds      
Variation            Control        Test        Control  Test
process_step                                                 
start              60.911733   56.240912           20.0  14.0
step_1             47.528433   58.481380           20.0  27.0
step_2             90.214112   88.133949           64.0  61.0
step_3            133.298306  124.832693           72.0  57.0

### KPI 2 Interpretation

The descriptive results show that the Test group does not have shorter interaction times at every process step.

Compared with the Control group:

- Test users progressed faster after `start`.
- Test users spent more time after reaching `step_1`.
- Test users progressed slightly faster after `step_2`.
- Test users progressed noticeably faster after `step_3`.

The median times show the same overall pattern and are considered the more robust measure because the duration distribution is right-skewed.

The sensitivity check also shows that including post-`confirm` activity affects the overall mean, particularly for the Test group. With `confirm` included, Test appears slightly slower overall (80.31 vs 79.57 seconds). After excluding `confirm`, Test has a lower overall mean time (75.94 vs 78.13 seconds).

Because `confirm` is the final process stage and has no required next step, it is excluded from the primary step-time KPI. Post-confirmation timing is retained separately as a sensitivity check.

Overall, the Test group shows shorter interaction times at most stages, while `step_1` shows the opposite pattern.

These results are descriptive and have not been statistically tested, so they should not be interpreted as confirmed effects of the redesigned interface.

## KPI 3 – Backward-Transition Rate (Error Proxy)

A backward transition is treated as a possible error or indicator of user friction, rather than a confirmed error.

The expected process order is:

`start → step_1 → step_2 → step_3 → confirm`

A transition is classified as backward when the next recorded step has a lower position than the current step within the same visit.

For example:

`step_2 → step_1`

is classified as a backward transition.

Because users may intentionally return to review or change information, backward navigation is used as an error proxy rather than interpreted as a confirmed user error.

The backward-transition rate is calculated as:

\[
\text{Backward-Transition Rate} =
\frac{\text{Number of backward transitions}}
{\text{Total valid transitions}}
\]

A lower backward-transition rate indicates less revisiting of earlier steps, although the reason for backward navigation cannot be determined from the available data.

In [48]:
# Assign a numeric order to each process step

step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

time_df["current_step_number"] = (
    time_df["process_step"].map(step_order)
)

time_df["next_step_number"] = (
    time_df["next_step"].map(step_order)
)

time_df["backward_transition"] = (
    time_df["next_step_number"]
    < time_df["current_step_number"]
)

time_df[
    [
        "visit_id",
        "Variation",
        "process_step",
        "next_step",
        "backward_transition"
    ]
].head(10)

,visit_id,Variation,process_step,next_step,backward_transition
0,100012776_37918976071_457913,Test,confirm,confirm,False
2,100019538_17884295066_43909,Test,start,step_1,False
3,100019538_17884295066_43909,Test,step_1,step_2,False
4,100019538_17884295066_43909,Test,step_2,step_1,True
5,100019538_17884295066_43909,Test,step_1,step_1,False
6,100019538_17884295066_43909,Test,step_1,start,True
7,100019538_17884295066_43909,Test,start,start,False
8,100019538_17884295066_43909,Test,start,step_1,False
9,100019538_17884295066_43909,Test,step_1,step_2,False
10,100019538_17884295066_43909,Test,step_2,step_3,False


In [50]:
# Calculate backward-transition rate for Test and Control

backward_summary = (
    time_df
    .groupby("Variation")
    .agg(
        total_transitions=("backward_transition", "count"),
        backward_transitions=("backward_transition", "sum")
    )
)

backward_summary["backward_transition_rate"] = (
    backward_summary["backward_transitions"]
    / backward_summary["total_transitions"]
)

backward_summary["backward_transition_rate_percent"] = (
    backward_summary["backward_transition_rate"] * 100
)

backward_summary

,total_transitions,backward_transitions,backward_transition_rate,backward_transition_rate_percent
Variation,,,,
Control,108402,9682,0.089316,8.931570
Test,139628,16358,0.117154,11.715415


### Compare backward-transition rates

The backward-transition rates of the Test and Control groups are compared using both the absolute difference and the relative difference.

This shows whether the redesigned interface is associated with more or less backward navigation than the original interface.

Because backward navigation is used as an error proxy, the comparison should be interpreted as a difference in revisiting behaviour or potential friction rather than confirmed navigation errors.

In [51]:
# Compare Test and Control backward-transition rates

test_backward_rate = backward_summary.loc[
    "Test",
    "backward_transition_rate"
]

control_backward_rate = backward_summary.loc[
    "Control",
    "backward_transition_rate"
]

absolute_difference = (
    test_backward_rate - control_backward_rate
)

relative_difference = (
    absolute_difference / control_backward_rate
)

print(
    f"Control backward-transition rate: "
    f"{control_backward_rate:.2%}"
)

print(
    f"Test backward-transition rate: "
    f"{test_backward_rate:.2%}"
)

print(
    f"Absolute difference: "
    f"{absolute_difference:.2%}"
)

print(
    f"Relative difference: "
    f"{relative_difference:.2%}"
)

Control backward-transition rate: 8.93%
Test backward-transition rate: 11.72%
Absolute difference: 2.78%
Relative difference: 31.17%


### KPI 3 Interpretation

The Test group shows a higher backward-transition rate than the Control group.

Compared with the Control group:

- Control backward-transition rate: 8.93%
- Test backward-transition rate: 11.72%

This represents an absolute increase of approximately 2.78 percentage points.

Descriptively, the Test experience is associated with more frequent backward navigation, while also achieving a higher overall completion rate.

Backward navigation may reflect users reviewing information, correcting an earlier choice or experiencing friction. The available event data does not allow us to determine the underlying reason.

The higher backward-transition rate should therefore be considered alongside the higher completion rate when evaluating the redesigned experience.

These differences are descriptive and have not yet been statistically tested.

# Phase 2 Summary

The descriptive KPI analysis identifies several differences between the Test and Control groups:

- **Overall completion:** Test achieved a higher completion rate (69.29%) than Control (65.58%), a difference of 3.71 percentage points.
- **Funnel progression:** The largest observed difference occurred at `start → step_1`, where Test conversion was approximately 4.86 percentage points higher.
- **Drop-off:** `step_1 → step_2` was the only transition where Test showed higher drop-off, by approximately 0.89 percentage points.
- **Time spent:** After excluding post-`confirm` activity, Test showed shorter median interaction times at most stages, while `step_1` showed the opposite pattern. The sensitivity check demonstrated that including `confirm` particularly affects the Test group's overall mean.
- **Backward transitions:** Test had a higher backward-transition rate (11.72%) than Control (8.93%).

Overall, the descriptive results show a mixed pattern. Test has higher completion and stronger progression through most of the funnel, but also more backward navigation and slightly weaker progression from `step_1 → step_2`.

These findings describe observed differences only. Phase 3 hypothesis testing is required to determine which differences are statistically significant.

# SUGGESTION: Phase 3 – Hypothesis Testing

## Hypothesis 3 – Step-Level Funnel Conversion

### Research Question

Does the redesigned interface change the probability that clients progress from one funnel stage to the next?

The overall completion KPI shows whether more clients ultimately complete the process. The step-level funnel analysis allows us to investigate **where** differences between Test and Control occur.

### Hypotheses

For each funnel transition:

**Null hypothesis (H₀):**

The step-to-step conversion rate is equal for Test and Control.

\[
H_0: p_{Test} = p_{Control}
\]

**Alternative hypothesis (H₁):**

The step-to-step conversion rate differs between Test and Control.

\[
H_1: p_{Test} \neq p_{Control}
\]

A two-proportion z-test will be performed for each transition:

- `start → step_1`
- `step_1 → step_2`
- `step_2 → step_3`
- `step_3 → confirm`

Because four comparisons are performed, a Bonferroni correction will be applied to account for multiple testing.

Significance level:

\[
\alpha = 0.05
\]

In [37]:
# Prepare the funnel data for Hypothesis 3

h3_df = funnel_summary[
    funnel_summary["process_step"] != "start"
].copy()

h3_df[
    [
        "Variation",
        "process_step",
        "previous_clients_reached",
        "clients_reached",
        "step_conversion_rate"
    ]
]

,Variation,process_step,previous_clients_reached,clients_reached,step_conversion_rate
1,Control,step_1,23526.0,20241,0.860367
2,Control,step_2,20241.0,18786,0.928116
3,Control,step_3,18786.0,17521,0.932663
4,Control,confirm,17521.0,15428,0.880543
6,Test,step_1,26961.0,24507,0.908980
7,Test,step_2,24507.0,22528,0.919248
8,Test,step_3,22528.0,21118,0.937411
9,Test,confirm,21118.0,18682,0.884648


In [39]:
# Compare Test and Control conversion rates at each funnel transition

hypothesis_3_results = []

transitions = ["step_1", "step_2", "step_3", "confirm"]

for step in transitions:

    control = h3_df[
        (h3_df["Variation"] == "Control") &
        (h3_df["process_step"] == step)
    ].iloc[0]

    test = h3_df[
        (h3_df["Variation"] == "Test") &
        (h3_df["process_step"] == step)
    ].iloc[0]

    # Number progressing to the next step
    successes = [
        test["clients_reached"],
        control["clients_reached"]
    ]

    # Number reaching the previous step
    totals = [
        test["previous_clients_reached"],
        control["previous_clients_reached"]
    ]

    z_stat, p_value = proportions_ztest(
        successes,
        totals,
        alternative="two-sided"
    )

    hypothesis_3_results.append({
        "transition": step,
        "control_conversion": control["step_conversion_rate"],
        "test_conversion": test["step_conversion_rate"],
        "difference": (
            test["step_conversion_rate"] -
            control["step_conversion_rate"]
        ),
        "z_statistic": z_stat,
        "p_value": p_value
    })

hypothesis_3_results = pd.DataFrame(hypothesis_3_results)

hypothesis_3_results

,transition,control_conversion,test_conversion,difference,z_statistic,p_value
0,step_1,0.860367,0.908980,0.048612,17.166186,4.756647e-66
1,step_2,0.928116,0.919248,-0.008869,-3.507971,4.515378e-04
2,step_3,0.932663,0.937411,0.004749,1.953070,5.081125e-02
3,confirm,0.880543,0.884648,0.004105,1.248737,2.117615e-01


### Correct for multiple comparisons

Four separate funnel transitions are tested.

Running multiple hypothesis tests increases the probability of finding a significant result by chance. Therefore, a Bonferroni correction is applied to control the overall significance level at 5%.

With four comparisons, the equivalent Bonferroni-adjusted significance threshold is:

\[
\alpha_{adjusted} = \frac{0.05}{4} = 0.0125
\]

In [40]:
# Apply Bonferroni correction to the four p-values

from statsmodels.stats.multitest import multipletests

reject, adjusted_p_values, _, _ = multipletests(
    hypothesis_3_results["p_value"],
    alpha=0.05,
    method="bonferroni"
)

hypothesis_3_results["adjusted_p_value"] = adjusted_p_values
hypothesis_3_results["significant_after_correction"] = reject

hypothesis_3_results

,transition,control_conversion,test_conversion,difference,z_statistic,p_value,adjusted_p_value,significant_after_correction
0,step_1,0.860367,0.908980,0.048612,17.166186,4.756647e-66,1.902659e-65,True
1,step_2,0.928116,0.919248,-0.008869,-3.507971,4.515378e-04,1.806151e-03,True
2,step_3,0.932663,0.937411,0.004749,1.953070,5.081125e-02,2.032450e-01,False
3,confirm,0.880543,0.884648,0.004105,1.248737,2.117615e-01,8.470458e-01,False


### Hypothesis 3 Conclusion

After applying the Bonferroni correction, two funnel transitions show statistically significant differences between the Test and Control groups.

**`start → step_1`**

- Control conversion: 86.04%
- Test conversion: 90.90%
- Difference: +4.86 percentage points
- Adjusted p-value < 0.001

The null hypothesis is rejected for this transition. The Test group has a significantly higher probability of progressing from `start` to `step_1`.

**`step_1 → step_2`**

- Control conversion: 92.81%
- Test conversion: 91.92%
- Difference: -0.89 percentage points
- Adjusted p-value ≈ 0.0018

The null hypothesis is also rejected for this transition. The Test group has a statistically significant, although relatively small, reduction in progression from `step_1` to `step_2`.

For `step_2 → step_3` and `step_3 → confirm`, the differences are not statistically significant after correcting for multiple comparisons.

### Business Interpretation

The higher overall completion rate of the Test group is not driven equally across the entire funnel.

The strongest improvement occurs at the beginning of the journey, where the redesigned interface substantially increases progression from `start` to `step_1`.

However, the following transition from `step_1` to `step_2` performs slightly worse in the Test group. This stage may therefore be a useful focus for a future UX investigation or targeted follow-up A/B test.

The later stages do not show statistically reliable differences between the two interfaces.


One important nuance: statistical significance does not mean the -0.89 pp difference at step_1 → step_2 is large or commercially important. The sample is large, so small differences can become significant. That's worth mentioning in the presentation.